# S04 — SQL for Data Science Patterns

**SQL is not just for extraction.** In modern data science, SQL is used to implement statistical computations, feature engineering pipelines, A/B test analysis, and ML-prep workflows — often entirely inside the database.

**Topics:** Feature engineering in SQL, statistical computations, A/B test analysis, funnel analysis, basket analysis (co-occurrence), sessionization, time-series feature generation, ML-ready data prep.

**Reference:** [DuckDB docs](https://duckdb.org/docs/)


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import fetch_openml

retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail['CustomerID'] = pd.to_numeric(retail['CustomerID'], errors='coerce')
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype(int)

# Synthetic A/B test data
np.random.seed(42)
n = 10000
ab = pd.DataFrame({
    'user_id': range(1, n+1),
    'group': np.random.choice(['control','treatment'], n, p=[0.5, 0.5]),
    'converted': np.where(
        np.random.choice(['control','treatment'], n, p=[0.5,0.5]) == 'treatment',
        np.random.binomial(1, 0.135, n),
        np.random.binomial(1, 0.12, n)
    ),
    'revenue': 0.0,
    'signup_date': pd.date_range('2023-01-01', periods=n, freq='1min')
})
ab.loc[ab['converted']==1, 'revenue'] = np.random.lognormal(3.5, 0.8, ab['converted'].sum())

con = duckdb.connect()
con.register('retail', retail)
con.register('ab_test', ab)

def q(sql): return con.execute(sql).df()
print(f"Ready. A/B test: {len(ab)} users")

---
## Exercise 1 — Feature Engineering in SQL

**Business question:** Build an ML-ready customer feature table entirely in SQL. For each customer compute:
- `recency_days`: days since last purchase from dataset max date
- `frequency`: number of unique invoices
- `monetary`: total revenue
- `avg_basket_size`: average items per invoice
- `avg_unit_price`: average unit price paid
- `pct_weekend_orders`: % of orders placed on Saturday or Sunday
- `tenure_days`: days from first to last purchase
- `revenue_trend`: slope of monthly revenue (last 3 months vs first 3 months, positive/negative)
- `log_monetary`: log(monetary + 1)
- `monetary_zscore`: (monetary - mean) / std across all customers

All in a single query. No Python post-processing.

In [ ]:
sql1 = """
-- YOUR SQL HERE
-- Use multiple CTEs and window functions for zscore
"""

result1 = q(sql1)
result1.head()

In [ ]:
# --- ASSERTIONS ---
required = ['recency_days','frequency','monetary','avg_basket_size',
            'avg_unit_price','pct_weekend_orders','tenure_days',
            'log_monetary','monetary_zscore']
cols_lower = result1.columns.str.lower().tolist()
for col in required:
    assert col in cols_lower, f"Missing feature: {col}"
assert (result1['recency_days'] >= 0).all()
assert result1['pct_weekend_orders'].between(0, 100).all()
assert abs(result1['monetary_zscore'].mean()) < 0.01, "Zscore mean must be ~0"
assert abs(result1['monetary_zscore'].std() - 1) < 0.01, "Zscore std must be ~1"
assert (result1['log_monetary'] >= 0).all()
print(f"✓ Exercise 1 passed — {len(result1)} customers, {len(result1.columns)} features")

---
## Exercise 2 — A/B Test Analysis in SQL

**Business question:** Analyze the A/B test using only SQL. Compute:
- Conversion rate per group
- Mean revenue per user (including non-converters)
- Relative lift in conversion rate
- Standard error of the conversion rate difference
- Z-score and approximate p-value for the two-proportion test
- A `significant` flag: TRUE if p_value < 0.05

**Hint:** p-value ≈ `2 * (1 - NORMAL_CDF(abs(z_score)))`. DuckDB has `erf()` — use the relationship `normal_cdf(x) = 0.5 * (1 + erf(x / sqrt(2)))`.

In [ ]:
sql2 = """
-- YOUR SQL HERE
-- Return a single summary row with all computed metrics
"""

result2 = q(sql2)
result2

In [ ]:
# --- ASSERTIONS (verify against scipy) ---
control = ab[ab['group']=='control']
treatment = ab[ab['group']=='treatment']
p_c = control['converted'].mean()
p_t = treatment['converted'].mean()
n_c, n_t = len(control), len(treatment)
se = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)
z = (p_t - p_c) / se
p_val = 2 * (1 - stats.norm.cdf(abs(z)))

cols_lower = result2.columns.str.lower().tolist()
z_col = [c for c in result2.columns if 'z' in c.lower() and 'score' in c.lower()]
if z_col:
    assert abs(float(result2[z_col[0]].iloc[0]) - z) < 0.1, "Z-score must match scipy"
sig_col = [c for c in result2.columns if 'sig' in c.lower()]
if sig_col:
    expected_sig = p_val < 0.05
    sql_sig = bool(result2[sig_col[0]].iloc[0])
    assert sql_sig == expected_sig
print(f"✓ Exercise 2 passed — Z={z:.3f}, p={p_val:.4f}, significant={p_val<0.05}")
print(result2.to_string(index=False))

---
## Exercise 3 — Funnel Analysis

**Business question:** Build a purchase funnel showing how many customers progressed through each stage and the drop-off at each step.

Define funnel stages:
1. **Visited**: any customer with at least 1 transaction
2. **Multi-product**: customers who bought ≥ 2 distinct products
3. **Repeat buyer**: customers with ≥ 2 distinct invoices
4. **High value**: customers with total revenue > median customer revenue
5. **Loyal**: customers with ≥ 3 months of activity

Return: stage, n_customers, pct_of_total, pct_of_previous_stage, drop_off_pct.

In [ ]:
sql3 = """
-- YOUR SQL HERE
"""

result3 = q(sql3)
result3

In [ ]:
# --- ASSERTIONS ---
assert len(result3) == 5, "Must have 5 funnel stages"
n_col = [c for c in result3.columns if 'n_' in c.lower() or 'count' in c.lower()][0]
assert result3[n_col].is_monotonic_decreasing, "Funnel must narrow at each stage"
pct_prev = [c for c in result3.columns if 'previous' in c.lower() or 'prev' in c.lower()]
if pct_prev:
    assert pd.isna(result3[pct_prev[0]].iloc[0]), "First stage has no previous"
print("✓ Exercise 3 passed")
print(result3.to_string(index=False))

---
## Exercise 4 — Basket Analysis: Product Co-occurrence

**Business question:** Find product pairs that are frequently bought together (market basket analysis). This powers recommendation engines.

For each pair of products (StockCode_A, StockCode_B) that appear in the same invoice:
- `co_occurrence_count`: number of invoices containing both
- `support`: co_occurrence / total_invoices
- `confidence_a_to_b`: P(B|A) = support(A,B) / support(A)
- `lift`: support(A,B) / (support(A) * support(B))

Filter: support > 0.01 and lift > 1.5. Return top 20 pairs by lift.

**Hint:** Self-join the invoice-product table on InvoiceNo where StockCode_A < StockCode_B (to avoid duplicates).

In [ ]:
sql4 = """
-- YOUR SQL HERE
-- This is computationally expensive — limit to top 100 products by frequency first
"""

result4 = q(sql4)
result4.head(10)

In [ ]:
# --- ASSERTIONS ---
assert len(result4) <= 20
for col in ['support','lift']:
    assert col in result4.columns, f"Missing: {col}"
assert (result4['support'] > 0.01).all()
assert (result4['lift'] > 1.5).all()
lift_col = result4['lift']
assert lift_col.is_monotonic_decreasing or len(result4) < 3
print(f"✓ Exercise 4 passed — {len(result4)} high-lift product pairs")
print(result4[['StockCode_A','StockCode_B','support','lift']].head())

---
## Exercise 5 — Sessionization

**Business question:** Define a customer "session" as a sequence of transactions where gaps between consecutive transactions are ≤ 30 days. Compute session-level metrics.

For each session:
- `session_id` (per customer, sequential from 1)
- `session_start`, `session_end`
- `session_duration_days`
- `session_revenue`
- `n_invoices_in_session`
- `session_type`: `'Single'` (1 invoice), `'Short'` (2-3), `'Long'` (4+)

Then aggregate: average session revenue by session_type.

In [ ]:
sql5 = """
-- YOUR SQL HERE
-- Hint: use LAG to find gaps, then cumulative SUM of gap flags to assign session_id
"""

result5 = q(sql5)
result5.head()

In [ ]:
# --- ASSERTIONS ---
assert len(result5) > 0
type_col = [c for c in result5.columns if 'type' in c.lower() or 'session_type' in c.lower()]
if type_col:
    assert set(result5[type_col[0]]).issubset({'Single','Short','Long'})
dur_col = [c for c in result5.columns if 'duration' in c.lower() or 'days' in c.lower()]
if dur_col:
    assert (result5[dur_col[0]] >= 0).all()
print(f"✓ Exercise 5 passed — {len(result5)} sessions")
if type_col:
    print(result5[type_col[0]].value_counts().to_string())

---
## Exercise 6 — Statistical Aggregates in SQL

**Business question:** Compute a full statistical profile of customer revenue using only SQL — the kind of profile you'd normally do in pandas/scipy.

Return a single row with:
- `n`, `mean`, `median`, `std`, `variance`
- `skewness`: `(mean - median) / std` (Pearson's approximation)
- `kurtosis`: use `KURTOSIS()` DuckDB aggregate
- `p10`, `p25`, `p75`, `p90`, `p99` (percentiles)
- `iqr`: p75 - p25
- `cv`: std / mean (coefficient of variation)
- `gini_approx`: `1 - 2 * SUM(cumulative_share * (1/n))` approximation

Verify each metric against Python/numpy.

In [ ]:
sql6 = """
-- YOUR SQL HERE
-- Compute customer-level revenue first, then aggregate
"""

result6 = q(sql6)
result6

In [ ]:
# --- ASSERTIONS vs Python reference ---
customer_rev = retail.groupby('CustomerID')['Revenue'].sum().values
assert abs(float(result6['mean'].iloc[0]) - customer_rev.mean()) < 1
assert abs(float(result6['median'].iloc[0]) - np.median(customer_rev)) < 1
assert abs(float(result6['std'].iloc[0]) - customer_rev.std(ddof=1)) < 1
assert abs(float(result6['p99'].iloc[0]) - np.percentile(customer_rev, 99)) < 10
print("✓ Exercise 6 passed — SQL stats match Python reference")
print(result6.to_string(index=False))

---
## Exercise 7 — Time Series Feature Generation

**Business question:** Generate lag and rolling features for time-series modeling, entirely in SQL.

For the monthly revenue table, create an ML-ready feature matrix with:
- `lag_1`, `lag_2`, `lag_3` (lagged revenues)
- `rolling_mean_3`, `rolling_mean_6` (moving averages)
- `rolling_std_3` (volatility)
- `mom_pct`, `yoy_pct` (period changes)
- `month_sin`, `month_cos` (cyclical encoding of month number: `sin(2π * month / 12)`, `cos(2π * month / 12)`)
- `is_q4`: boolean, TRUE for October–December
- `target`: next month's revenue (LEAD 1)

Drop rows where any lag or rolling feature is NULL (first few rows).

In [ ]:
monthly = retail.groupby('Month').agg(Revenue=('Revenue','sum')).reset_index()
monthly['month_num'] = pd.to_datetime(monthly['Month'].astype(str)).dt.month
con.register('monthly', monthly)

sql7 = """
-- YOUR SQL HERE
"""

result7 = q(sql7)
result7

In [ ]:
# --- ASSERTIONS ---
assert result7.isna().sum().sum() == 0, "No NULLs allowed in ML feature matrix"
for col in ['lag_1','lag_2','lag_3','rolling_mean_3','mom_pct','month_sin','month_cos','is_q4','target']:
    assert col in result7.columns, f"Missing: {col}"
# Cyclical features must be in [-1, 1]
assert result7['month_sin'].between(-1, 1).all()
assert result7['month_cos'].between(-1, 1).all()
# target = next month Revenue
assert result7['target'].iloc[0] == pytest_approx_sql(result7['Revenue'].iloc[1] if 'Revenue' in result7.columns else None)
print(f"✓ Exercise 7 passed — {len(result7)} rows, {len(result7.columns)} features")

In [ ]:
# Simpler target check (no external function needed)
for col in ['lag_1','lag_2','lag_3','rolling_mean_3','mom_pct','month_sin','month_cos','is_q4','target']:
    assert col in result7.columns, f"Missing: {col}"
assert result7.isna().sum().sum() == 0
print(f"✓ Exercise 7 passed — {len(result7)} rows, {len(result7.columns)} features")
print(result7.head())

---
## Exercise 8 — Cross-Validation Splits in SQL

**Business question:** Implement time-series cross-validation splits in SQL — a pattern used when you need reproducible train/test splits computed at the database layer.

For the monthly feature matrix:
1. Assign each row to a fold (1–5) using a sequential approach: fold 1 = first 20% of months, etc.
2. For each fold, the training set = all months before the fold, test set = the fold itself
3. Return: fold_number, n_train_months, n_test_months, train_start, train_end, test_start, test_end

In [ ]:
sql8 = """
-- YOUR SQL HERE
"""

result8 = q(sql8)
result8

In [ ]:
# --- ASSERTIONS ---
assert len(result8) == 5, "Must have 5 folds"
fold_col = result8.columns[0]
assert list(result8[fold_col]) == [1,2,3,4,5]
n_train = [c for c in result8.columns if 'train' in c.lower() and 'n_' in c.lower()][0]
assert result8[n_train].is_monotonic_increasing, "Training set must grow with each fold"
print("✓ Exercise 8 passed")
print(result8.to_string(index=False))

---
## Exercise 9 — Sampling Strategies in SQL

**Business question:** Implement three sampling strategies in SQL:
1. **Random sample**: 10% of customers using `USING SAMPLE 10%`
2. **Stratified sample**: 10% per country (top 10 countries), maintaining proportions
3. **Systematic sample**: every Nth customer (based on row_number mod N = 0)

For each sample, verify that:
- Size is approximately correct
- Stratified sample preserves country proportions within 5%

In [ ]:
sql9_random = """
-- YOUR SQL HERE — random sample
"""

sql9_stratified = """
-- YOUR SQL HERE — stratified sample
"""

sql9_systematic = """
-- YOUR SQL HERE — systematic sample (every 10th customer)
"""

r_random = q(sql9_random)
r_strat = q(sql9_stratified)
r_sys = q(sql9_systematic)
print(f"Random: {len(r_random)} | Stratified: {len(r_strat)} | Systematic: {len(r_sys)}")

In [ ]:
# --- ASSERTIONS ---
total_customers = retail['CustomerID'].nunique()
# Random: roughly 10% of customers
assert 0.05 * total_customers <= len(r_random) <= 0.20 * total_customers, \
    "Random sample should be ~10% of customers"

# Stratified: check proportions preserved
if 'Country' in r_strat.columns:
    top10 = retail.groupby('Country')['CustomerID'].nunique().nlargest(10)
    top10_pct = top10 / top10.sum()
    sample_pct = r_strat['Country'].value_counts(normalize=True)
    for country in top10_pct.index:
        if country in sample_pct.index:
            diff = abs(top10_pct[country] - sample_pct[country])
            assert diff < 0.10, f"Country {country} proportion off by {diff:.2%}"

print("✓ Exercise 9 passed")

---
## Exercise 10 — Capstone: End-to-End ML Prep Pipeline

**Spec:** Build a complete ML training dataset using only SQL — from raw retail to a model-ready feature matrix.

Pipeline:
1. **Clean**: remove nulls, invalid quantities/prices
2. **Customer features**: RFM + behavioral features (10+ columns)
3. **Target**: binary `will_churn` = 1 if customer has no purchases in the last 90 days of the dataset
4. **Stratified train/test split** in SQL: 80/20 based on CustomerID hash
5. **Feature statistics**: compute mean and std of each numeric feature on train set only
6. **Normalize**: apply `(value - train_mean) / train_std` to both sets using train statistics
7. Write `train_features` and `test_features` tables
8. Validate: no nulls, class balance within 5% between train and test, correct row counts

This is the SQL version of a sklearn Pipeline.

In [ ]:
# YOUR PIPELINE HERE — use multiple CTE blocks
pipeline_sql = """
-- Step 1: Clean
-- Step 2: Customer features
-- Step 3: Target
-- Step 4: Split
-- Step 5-6: Normalize
-- Step 7: Create tables
"""

con.execute(pipeline_sql)

train = q("SELECT * FROM train_features")
test = q("SELECT * FROM test_features")
print(f"Train: {train.shape} | Test: {test.shape}")
train.head()

In [ ]:
# --- ASSERTIONS ---
total = len(train) + len(test)
# 80/20 split
assert 0.75 <= len(train)/total <= 0.85, "Train must be ~80%"
# No nulls
assert train.isna().sum().sum() == 0, "No nulls in train"
assert test.isna().sum().sum() == 0, "No nulls in test"
# Target column present
assert 'will_churn' in train.columns
assert 'will_churn' in test.columns
# Class balance within 5%
train_churn = train['will_churn'].mean()
test_churn = test['will_churn'].mean()
assert abs(train_churn - test_churn) < 0.05, "Class balance must be similar"
# Normalized features: train mean ~0 (for numeric cols except target)
num_cols = train.select_dtypes(include='number').columns.drop('will_churn', errors='ignore')
for col in num_cols[:3]:  # check first 3 numeric cols
    assert abs(train[col].mean()) < 0.1, f"Train {col} not normalized (mean={train[col].mean():.4f})"

print(f"✓ Exercise 10 passed")
print(f"Train: {len(train)} rows | Test: {len(test)} rows")
print(f"Churn rate — Train: {train_churn:.2%} | Test: {test_churn:.2%}")